# 02 — BERT to ONNX Export

## Objective
Export a fine-tuned BERT sequence classification model from PyTorch to ONNX format. Understand computational graphs, dynamic input dimensions, and model validation. Verify the exported model's structure and prepare it for inference with ONNX Runtime.

### Install required dependencies

In [ ]:
%pip install onnx onnxruntime optimum-onnx

### Exporting a model

#### Configuration step

In [ ]:
from pathlib import Path

MODEL_ID = "textattack/bert-base-uncased-SST-2"

print(Path.cwd())
EXPORT_DIR = Path.cwd().parent / "models" / "bert-sst2-onnx"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

#### Export step

In [ ]:
from optimum.exporters.onnx import main_export

main_export(
    model_name_or_path=MODEL_ID,
    output=EXPORT_DIR,
    task="text-classification",
    opset=18
)

**What is opset?**

ONNX use the operators which represents calculations like:
- MathMul
- Add
- Softmax
- Reshape

Opset define which version of these operators use in this scenario.

> Opset is not a version of BERT or ONNX Runtime.

### Exported files

We use a iterdir() method for iterate on catalog elements to show our exported files, like:
- model.onnx
- config.json
- tokenizer.json
- tokenizer_config.json
- ...

A list of files depends on a version of using library and export configuration 

In [3]:
for file in EXPORT_DIR.iterdir():
    print(file.name)

vocab.txt
tokenizer_config.json
config.json
special_tokens_map.json
tokenizer.json


### Validate the ONNX structure

We have to check if generated file are corrected, what means that ONNX can open correctly and read the structure or not.

In [ ]:
import onnx

onnx_path = EXPORT_DIR / "model.onnx"
onnx_model = onnx.load(str(onnx_path))

print(onnx.checker.check_model(onnx_model, full_check=True))

**Important facts**

`onnx.load()` method load onnx model to memory as `ModelProto` object.

>ModelProto is a representation of the model based on buffers protocol. Consist of graphs, metadata and information about operators. 

### Model I/O Inspection

In [ ]:
graph = onnx_model.graph

print("INPUTS:")

for input_tensor in graph.input:
    print(input_tensor.name)

print("\nOUTPUTS:")

for output_tensor in graph.output:
    print(output_tensor.name)

### Tensor Shapes

In [ ]:
def show_tensor_shape(tensor):
    dimensions = tensor.type.tensor_type.shape.dim

    shape = [
        dim.dim_param if dim.dim_param else dim.dim_value
        for dim in dimensions
    ]

    print(f"{tensor.name}: {shape}")

for tensor in graph.input:
    show_tensor_shape(tensor)

for tensor in graph.output:
    show_tensor_shape(tensor)